
# Roxy notebook example: Sliding-window descriptors

This notebook is a **reference implementation example** for the **sliding-window descriptor family** in Roxy.

Sliding-window descriptors summarize how local properties vary along the sequence by scanning overlapping windows and aggregating the resulting profiles.

## Covered outputs

This notebook implements examples such as:

- windowed hydrophobicity
- windowed polarity
- windowed charge fraction
- windowed aromatic fraction
- windowed entropy
- mean / std / min / max of window profiles
- amplitude (max - min) of window profiles
- fraction of windows above thresholds
- N-to-C trend proxies from window profiles
- class-style implementation for later migration into Roxy

These descriptors are useful because they capture **local heterogeneity** and **regional trends** that are lost in global averages.


In [28]:

from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [29]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "win_1",
            "win_2",
            "win_3",
            "win_4",
            "win_5",
            "win_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,win_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,win_2,GGGGGGGGGGGGGGG,B
2,win_3,KRRKRRKRRKRRDDDDEE,A
3,win_4,ACDEFGHIKLMNPQRSTVWY,B
4,win_5,PPPPGSSSSSTTTTNNQQQ,A
5,win_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [30]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

POLARITY = {
    "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
    "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
    "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
    "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
}

AA_GROUPS = {
    "charged": set("KRHDE"),
    "positive": set("KRH"),
    "negative": set("DE"),
    "aromatic": set("FWYH"),
}


## Helper functions

In [31]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    probs = np.array([count / len(seq) for count in counts.values()], dtype=float)
    return float(-(probs * np.log2(probs)).sum())


def scale_mean(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.mean([scale[aa] for aa in seq]))


def group_fraction(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def profile_stats(values):
    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "amplitude": np.nan,
            "start_end_diff": np.nan,
        }
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "amplitude": float(np.max(values) - np.min(values)),
        "start_end_diff": float(values[-1] - values[0]),
    }


def fraction_above_threshold(values, threshold: float) -> float:
    if len(values) == 0:
        return np.nan
    return float(np.mean(np.array(values) > threshold))


## Core sliding-window descriptor function

In [32]:

def sliding_window_descriptors(seq: str, window_sizes=(5, 7, 9)) -> dict:
    seq = clean_sequence(seq)

    out = {
        "win_length": len(seq),
        "win_valid_residue_count": len(seq),
    }

    for window_size in window_sizes:
        ws = windows(seq, window_size)

        hydro_profile = [scale_mean(w, HYDROPATHY) for w in ws]
        polarity_profile = [scale_mean(w, POLARITY) for w in ws]
        charged_profile = [group_fraction(w, AA_GROUPS["charged"]) for w in ws]
        aromatic_profile = [group_fraction(w, AA_GROUPS["aromatic"]) for w in ws]
        entropy_profile = [shannon_entropy(w) for w in ws]

        profile_map = {
            "hydropathy": hydro_profile,
            "polarity": polarity_profile,
            "charged_frac": charged_profile,
            "aromatic_frac": aromatic_profile,
            "entropy": entropy_profile,
        }

        for profile_name, values in profile_map.items():
            stats = profile_stats(values)
            prefix = f"win{window_size}_{profile_name}"

            out[f"{prefix}_mean"] = stats["mean"]
            out[f"{prefix}_std"] = stats["std"]
            out[f"{prefix}_min"] = stats["min"]
            out[f"{prefix}_max"] = stats["max"]
            out[f"{prefix}_amplitude"] = stats["amplitude"]
            out[f"{prefix}_start_end_diff"] = stats["start_end_diff"]

        out[f"win{window_size}_hydropathy_high_fraction"] = fraction_above_threshold(hydro_profile, threshold=1.0)
        out[f"win{window_size}_charged_high_fraction"] = fraction_above_threshold(charged_profile, threshold=0.4)
        out[f"win{window_size}_aromatic_high_fraction"] = fraction_above_threshold(aromatic_profile, threshold=0.2)
        out[f"win{window_size}_low_entropy_fraction"] = fraction_above_threshold([-v for v in entropy_profile], threshold=-1.5)

    return out


## Functional usage on one sequence

In [33]:

example = sliding_window_descriptors(df_demo.loc[0, "sequence"], window_sizes=(5, 7))
list(example.items())[:18]


[('win_length', 24),
 ('win_valid_residue_count', 24),
 ('win5_hydropathy_mean', 0.999),
 ('win5_hydropathy_std', 1.4189781534611445),
 ('win5_hydropathy_min', -1.1199999999999999),
 ('win5_hydropathy_max', 3.4),
 ('win5_hydropathy_amplitude', 4.52),
 ('win5_hydropathy_start_end_diff', -0.6000000000000002),
 ('win5_polarity_mean', 7.123),
 ('win5_polarity_std', 1.0714714181908915),
 ('win5_polarity_min', 5.02),
 ('win5_polarity_max', 8.639999999999999),
 ('win5_polarity_amplitude', 3.619999999999999),
 ('win5_polarity_start_end_diff', 0.8400000000000007),
 ('win5_charged_frac_mean', 0.1),
 ('win5_charged_frac_std', 0.1341640786499874),
 ('win5_charged_frac_min', 0.0),
 ('win5_charged_frac_max', 0.4)]

## Apply sliding-window descriptors to the full dataset

In [34]:

df_win = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(lambda x: sliding_window_descriptors(x, window_sizes=(5, 7, 9))).apply(pd.Series),
    ],
    axis=1,
)

df_win.head()


,sequence_id,sequence,label,win_length,win_valid_residue_count,win5_hydropathy_mean,win5_hydropathy_std,win5_hydropathy_min,win5_hydropathy_max,win5_hydropathy_amplitude,...,win9_entropy_mean,win9_entropy_std,win9_entropy_min,win9_entropy_max,win9_entropy_amplitude,win9_entropy_start_end_diff,win9_hydropathy_high_fraction,win9_charged_high_fraction,win9_aromatic_high_fraction,win9_low_entropy_fraction
0,win_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.999000,1.418978,-1.12,3.40,4.52,...,2.447160,0.374570,1.891061,3.169925,1.278864,-0.528321,0.625,0.00,1.0,0.000000
1,win_2,GGGGGGGGGGGGGGG,B,15.0,15.0,-0.400000,0.000000,-0.40,-0.40,0.00,...,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000,0.00,0.0,1.000000
2,win_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,-4.085714,0.304859,-4.38,-3.50,0.88,...,1.284505,0.340437,0.918296,1.836592,0.918296,0.918296,0.000,1.00,0.0,0.700000
3,win_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,-0.662500,1.005805,-2.78,0.62,3.40,...,3.169925,0.000000,3.169925,3.169925,0.000000,0.000000,0.000,0.25,0.5,0.000000
4,win_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,-1.393333,0.852187,-3.50,-0.72,2.78,...,1.502574,0.234410,0.991076,1.836592,0.845516,0.138346,0.000,0.00,0.0,0.454545


## Inspect sliding-window descriptor columns

In [35]:

win_cols = [c for c in df_win.columns if c.startswith("win") and c not in {"win_length", "win_valid_residue_count"}]
len(win_cols), win_cols


(102,
 ['win5_hydropathy_mean',
  'win5_hydropathy_std',
  'win5_hydropathy_min',
  'win5_hydropathy_max',
  'win5_hydropathy_amplitude',
  'win5_hydropathy_start_end_diff',
  'win5_polarity_mean',
  'win5_polarity_std',
  'win5_polarity_min',
  'win5_polarity_max',
  'win5_polarity_amplitude',
  'win5_polarity_start_end_diff',
  'win5_charged_frac_mean',
  'win5_charged_frac_std',
  'win5_charged_frac_min',
  'win5_charged_frac_max',
  'win5_charged_frac_amplitude',
  'win5_charged_frac_start_end_diff',
  'win5_aromatic_frac_mean',
  'win5_aromatic_frac_std',
  'win5_aromatic_frac_min',
  'win5_aromatic_frac_max',
  'win5_aromatic_frac_amplitude',
  'win5_aromatic_frac_start_end_diff',
  'win5_entropy_mean',
  'win5_entropy_std',
  'win5_entropy_min',
  'win5_entropy_max',
  'win5_entropy_amplitude',
  'win5_entropy_start_end_diff',
  'win5_hydropathy_high_fraction',
  'win5_charged_high_fraction',
  'win5_aromatic_high_fraction',
  'win5_low_entropy_fraction',
  'win7_hydropathy_mean

In [36]:
df_win[
    [
        "sequence_id",
        "win5_hydropathy_mean",
        "win5_hydropathy_std",
        "win5_hydropathy_amplitude",
        "win7_entropy_mean",
        "win7_entropy_amplitude",
        "win9_charged_high_fraction",
    ]
]


,sequence_id,win5_hydropathy_mean,win5_hydropathy_std,win5_hydropathy_amplitude,win7_entropy_mean,win7_entropy_amplitude,win9_charged_high_fraction
0,win_1,0.999000,1.418978,4.52,2.251489,1.250698,0.000000
1,win_2,-0.400000,0.000000,0.00,0.000000,0.000000,0.000000
2,win_3,-4.085714,0.304859,0.88,1.147139,0.585695,1.000000
3,win_4,-0.662500,1.005805,3.40,2.807355,0.000000,0.250000
4,win_5,-1.393333,0.852187,2.78,1.275946,0.693536,0.000000
5,win_6,-1.308235,1.102512,4.00,2.635926,0.285714,0.384615


## Dataset-level summary

In [37]:

win_summary = (
    df_win[win_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

win_summary.head(15)


,descriptor,mean_value
0,win5_polarity_max,10.443333
1,win7_polarity_max,10.100000
2,win9_polarity_max,9.672222
3,win5_polarity_mean,9.093767
4,win7_polarity_mean,9.063435
5,win9_polarity_mean,9.026743
6,win9_polarity_min,8.492593
7,win7_polarity_min,8.278571
8,win5_polarity_min,7.980000
9,win5_hydropathy_amplitude,2.596667


## Sanity checks

In [38]:

assert "win5_hydropathy_mean" in df_win.columns
assert "win7_polarity_std" in df_win.columns
assert df_win["win_length"].min() > 0

print(f"Number of sliding-window descriptor columns: {len(win_cols)}")
print("Sliding-window descriptor checks passed.")


Number of sliding-window descriptor columns: 102
Sliding-window descriptor checks passed.


## Class-style implementation closer to the real package

In [39]:

class SlidingWindowDescriptors:
    """Example class-style sliding-window implementation for later migration into Roxy."""

    def __init__(self, window_sizes=(5, 7, 9)):
        self.window_sizes = tuple(window_sizes)

    def transform_sequence(self, seq: str) -> dict:
        return sliding_window_descriptors(seq, window_sizes=self.window_sizes)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


win_transformer = SlidingWindowDescriptors(window_sizes=(5, 7, 9))
win_matrix = win_transformer.transform(df_demo["sequence"].tolist())
win_matrix.head()


,win_length,win_valid_residue_count,win5_hydropathy_mean,win5_hydropathy_std,win5_hydropathy_min,win5_hydropathy_max,win5_hydropathy_amplitude,win5_hydropathy_start_end_diff,win5_polarity_mean,win5_polarity_std,...,win9_entropy_mean,win9_entropy_std,win9_entropy_min,win9_entropy_max,win9_entropy_amplitude,win9_entropy_start_end_diff,win9_hydropathy_high_fraction,win9_charged_high_fraction,win9_aromatic_high_fraction,win9_low_entropy_fraction
0,24,24,0.999000,1.418978,-1.12,3.40,4.52,-0.60,7.123000,1.071471,...,2.447160,0.374570,1.891061,3.169925,1.278864,-0.528321,0.625,0.00,1.0,0.000000
1,15,15,-0.400000,0.000000,-0.40,-0.40,0.00,0.00,9.000000,0.000000,...,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000,0.00,0.0,1.000000
2,18,18,-4.085714,0.304859,-4.38,-3.50,0.88,0.76,11.355714,0.795817,...,1.284505,0.340437,0.918296,1.836592,0.918296,0.918296,0.000,1.00,0.0,0.700000
3,20,20,-0.662500,1.005805,-2.78,0.62,3.40,0.08,8.548750,0.810277,...,3.169925,0.000000,3.169925,3.169925,0.000000,0.000000,0.000,0.25,0.5,0.000000
4,19,19,-1.393333,0.852187,-3.50,-0.72,2.78,-2.14,9.258667,0.753037,...,1.502574,0.234410,0.991076,1.836592,0.845516,0.138346,0.000,0.00,0.0,0.454545


## Merge transformer output back to the dataset

In [40]:

df_win_class = pd.concat([df_demo, win_matrix], axis=1)
df_win_class.head()


,sequence_id,sequence,label,win_length,win_valid_residue_count,win5_hydropathy_mean,win5_hydropathy_std,win5_hydropathy_min,win5_hydropathy_max,win5_hydropathy_amplitude,...,win9_entropy_mean,win9_entropy_std,win9_entropy_min,win9_entropy_max,win9_entropy_amplitude,win9_entropy_start_end_diff,win9_hydropathy_high_fraction,win9_charged_high_fraction,win9_aromatic_high_fraction,win9_low_entropy_fraction
0,win_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.999000,1.418978,-1.12,3.40,4.52,...,2.447160,0.374570,1.891061,3.169925,1.278864,-0.528321,0.625,0.00,1.0,0.000000
1,win_2,GGGGGGGGGGGGGGG,B,15,15,-0.400000,0.000000,-0.40,-0.40,0.00,...,0.000000,0.000000,-0.000000,-0.000000,0.000000,0.000000,0.000,0.00,0.0,1.000000
2,win_3,KRRKRRKRRKRRDDDDEE,A,18,18,-4.085714,0.304859,-4.38,-3.50,0.88,...,1.284505,0.340437,0.918296,1.836592,0.918296,0.918296,0.000,1.00,0.0,0.700000
3,win_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,-0.662500,1.005805,-2.78,0.62,3.40,...,3.169925,0.000000,3.169925,3.169925,0.000000,0.000000,0.000,0.25,0.5,0.000000
4,win_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,-1.393333,0.852187,-3.50,-0.72,2.78,...,1.502574,0.234410,0.991076,1.836592,0.845516,0.138346,0.000,0.00,0.0,0.454545



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/windows.py`
- keep residue groups and scales in `roxy/core/constants.py`
- expose a class such as `SlidingWindowDescriptors`
- allow configurable:
  - window sizes
  - local properties to profile
  - profile summary statistics
  - thresholds for "high" window fractions
- add tests for:
  - empty sequences
  - sequences shorter than the chosen window
  - highly heterogeneous sequences
  - low-complexity sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [41]:
# df_win.to_csv("demo_sliding_window_descriptors.csv", index=False)
